In [1]:
import pandas as pd
import numpy as np
import sys
from tqdm.auto import tqdm
from antropy import perm_entropy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, lowpass_filter_causal, bootstrap_report

/home/marco/git/online-impairment-detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sampling_rate = 100 # Hz, sensor sampling frequency
lpf_cutoff = 10 # Hz, FIR lowpass cutoff
lpf_order = 64 # FIR filter taps
pe_order = 5 # permutation entropy embedding dimension
pe_delay = int(sampling_rate / (2 * lpf_cutoff)) # PE delay (Nyquist-matched)
pe_w = 1000 # entropy window size in samples
pe_s = 100 # window step size in samples
alpha = 0.023 # false-alarm bound
k = 10 # shrinkage strength: pseudo-windows pulling the running mean toward the sober baseline
glue = False # set True to glue metrics for result collection
per_ride_path = None # set a path to save per-ride outcomes as CSV

In [3]:
# Parameters
alpha = 0.023
glue = True
per_ride_path = "/home/marco/git/online-impairment-detection/output/avg_cal_lr_alpha_0.023.csv"


In [4]:
data = pd.read_parquet("../data/data.parquet")
data

,ride_id,time_index,participant_id,run,trial,ax,ay,az,rx,ry,rz,throttle,brake_l,brake_r
0,000e5ab725f4,0,23.0,Sober,2.0,-0.442073,-9.215956,-3.105285,0.020311,-0.136070,0.213650,41.0,41.0,41.0
1,000e5ab725f4,10,23.0,Sober,2.0,-0.547358,-9.390632,-2.988635,0.019853,-0.149967,0.210443,41.0,41.0,41.0
2,000e5ab725f4,20,23.0,Sober,2.0,-0.552742,-9.348757,-2.974876,0.017715,-0.165086,0.208457,41.0,41.0,41.0
3,000e5ab725f4,30,23.0,Sober,2.0,-0.605384,-9.287740,-3.065803,0.014508,-0.182190,0.206319,41.0,41.0,41.0
4,000e5ab725f4,40,23.0,Sober,2.0,-0.631106,-9.226125,-3.071786,0.014966,-0.197767,0.204181,41.0,41.0,41.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
876191,fdc39ec37c87,67410,13.0,Sober,2.0,1.475773,-10.374680,-0.763908,-0.276263,1.067486,0.098043,147.0,41.0,41.0
876192,fdc39ec37c87,67420,13.0,Sober,2.0,1.175474,-9.505487,-1.128814,-0.290924,1.067792,0.093462,147.0,41.0,41.0
876193,fdc39ec37c87,67430,13.0,Sober,2.0,1.273579,-10.332806,-0.802791,-0.313373,1.102611,0.105374,147.0,41.0,41.0
876194,fdc39ec37c87,67440,13.0,Sober,2.0,1.231107,-9.077770,-1.895115,-0.332616,1.102306,0.106443,147.0,41.0,41.0


In [5]:
labels = data.groupby(["ride_id", "participant_id"])["run"].first().reset_index()
labels

,ride_id,participant_id,run
0,000e5ab725f4,23.0,Sober
1,02a49cf28afc,31.0,High
2,04a58ca54c11,28.0,Sober
3,065ea8ad5f24,23.0,Low
4,097169d2138f,10.0,Low
...,...,...,...
136,fbf0250f1611,8.0,High
137,fc09a596ffcd,25.0,High
138,fcb8375a05a2,24.0,Low
139,fd0f1af171a2,23.0,Low


In [6]:
features = ["ax", "ay", "az", "rx", "ry", "rz", "throttle"]

data_np = reshape_to_numpy(
    data,
    features = features,
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_lpf = lowpass_filter_causal(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=lpf_cutoff,
    gyro_cutoff=lpf_cutoff,
    order=lpf_order,
    fs=sampling_rate,
)

In [7]:
# Precompute windowed entropies for all rides.
n_channels = len(features)
all_windowed_ent = []

for ride in tqdm(data_np_lpf, desc="windowed entropy"):
    # get row indices without any NaN
    valid = np.flatnonzero(~np.any(np.isnan(ride), axis=1))
    # strip leading (FIR warm-up) and trailing NaNs
    ride_trimmed = ride[valid[0]:valid[-1] + 1]
    # sliding-window permutation entropy per channel
    n_steps = max(0, (len(ride_trimmed) - pe_w) // pe_s + 1)
    ride_ent = np.empty((n_steps, n_channels))
    for t in range(n_steps):
        window = ride_trimmed[t * pe_s : t * pe_s + pe_w]
        for j in range(n_channels):
            ride_ent[t, j] = perm_entropy(window[:, j],
                                          order=pe_order, delay=pe_delay, normalize=True)
    all_windowed_ent.append(ride_ent)

windowed entropy:   0%|                                                                                                                                                         | 0/141 [00:00<?, ?it/s]

windowed entropy:   1%|██                                                                                                                                               | 2/141 [00:00<00:09, 15.41it/s]

windowed entropy:   3%|████                                                                                                                                             | 4/141 [00:00<00:10, 13.48it/s]

windowed entropy:   4%|██████▏                                                                                                                                          | 6/141 [00:00<00:10, 13.08it/s]

windowed entropy:   6%|████████▏                                                                                                                                        | 8/141 [00:00<00:09, 14.13it/s]

...

windowed entropy:  95%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉       | 134/141 [00:08<00:00, 15.33it/s]

windowed entropy:  96%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉     | 136/141 [00:08<00:00, 15.39it/s]

windowed entropy:  98%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉   | 138/141 [00:08<00:00, 15.67it/s]

windowed entropy:  99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉ | 140/141 [00:08<00:00, 15.97it/s]

windowed entropy: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 141/141 [00:08<00:00, 15.86it/s]

In [8]:
# Within-subject centering (WSC): subtract per-channel mean of the
# participant's other rides to remove between-subject variability
# without needing to know which ride is sober.
pids = labels["participant_id"].values
all_windowed_wsc = []

for i in range(len(labels)): # i is a positional index, not a label
    # indices of the same participant's other rides
    same_pid = np.flatnonzero(pids == pids[i])
    other_idx = same_pid[same_pid != i]
    # collect their entropy windows
    other_windows = [all_windowed_ent[j] for j in other_idx]
    # per-channel mean across those rides
    mu = np.nanmean(np.vstack(other_windows), axis=0)
    # subtract to center within subject
    all_windowed_wsc.append(all_windowed_ent[i] - mu)

In [9]:
# LOPO CV
logo = LeaveOneGroupOut()
results = []

for train_idx, test_idx in tqdm(list(logo.split(labels, labels["run"], labels["participant_id"])), desc="LOPO"):
    # split training windows by label
    train_labels = labels.iloc[train_idx].reset_index(drop=True)
    tr_sober = [all_windowed_wsc[train_idx[j]] for j in range(len(train_labels)) if train_labels["run"].iloc[j] == "Sober"]
    tr_imp   = [all_windowed_wsc[train_idx[j]] for j in range(len(train_labels)) if train_labels["run"].iloc[j] != "Sober"]

    # fit a single logistic regression; P(impaired | x) as detector score
    X_tr = np.vstack(tr_sober + tr_imp)
    y_tr = np.concatenate([np.zeros(sum(map(len, tr_sober))), np.ones(sum(map(len, tr_imp)))])
    clf = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)

    # sober baseline score: shrinkage target for the running mean (no test leakage)
    m0 = clf.predict_proba(np.vstack(tr_sober))[:, 1].mean()

    # compute running-mean LR path for held-out participant
    for ride_idx in test_idx:
        # load test ride windows
        ride_wsc = all_windowed_wsc[ride_idx]
        n_windows = len(ride_wsc)

        # init running-mean state
        sum_lr = 0.0 # cumulative LR score
        mean_path = [] # running mean at each step

        # sequential running mean of LR probabilities
        for t in range(n_windows):
            # get current window
            x = ride_wsc[t:t+1]

            # LR score
            s_t = clf.predict_proba(x)[0, 1]

            # shrinkage running-mean update: k pseudo-windows pull early estimates
            # toward the sober baseline m0, damping small-sample spikes
            sum_lr += s_t
            mean_path.append((sum_lr + k * m0) / (t + 1 + k))

        results.append({
            "run": labels["run"].iloc[ride_idx],
            "participant_id": labels["participant_id"].iloc[ride_idx],
            "mean_path": mean_path,
        })

df_res = pd.DataFrame(results)
df_res

LOPO:   0%|                                                                                                                                                                      | 0/25 [00:00<?, ?it/s]

LOPO:   8%|████████████▋                                                                                                                                                 | 2/25 [00:00<00:01, 18.49it/s]

LOPO:  16%|█████████████████████████▎                                                                                                                                    | 4/25 [00:00<00:01, 14.03it/s]

LOPO:  24%|█████████████████████████████████████▉                                                                                                                        | 6/25 [00:00<00:01, 13.38it/s]

LOPO:  32%|██████████████████████████████████████████████████▌                                                                                                           | 8/25 [00:00<00:01, 14.08it/s]

...

LOPO:  80%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                               | 20/25 [00:01<00:00, 10.71it/s]

LOPO:  88%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                  | 22/25 [00:01<00:00, 10.20it/s]

LOPO:  96%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋      | 24/25 [00:02<00:00,  9.98it/s]

LOPO: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:02<00:00, 11.82it/s]

,run,participant_id,mean_path
0,High,3.0,"[0.4782833819480143, 0.5176180617936335, 0.553..."
1,Sober,3.0,"[0.46058195079816233, 0.48201164664443574, 0.4..."
2,High,3.0,"[0.4804965395454593, 0.5221958112488587, 0.556..."
3,Low,3.0,"[0.4538851868405219, 0.46681271101042715, 0.48..."
4,Low,3.0,"[0.47409700322973053, 0.5120840505749872, 0.54..."
...,...,...,...
136,Sober,31.0,"[0.4337333199146217, 0.42564298076245216, 0.42..."
137,Low,31.0,"[0.4874349185638229, 0.521108561581999, 0.5465..."
138,Low,31.0,"[0.44833040445940886, 0.45153349454108144, 0.4..."
139,Sober,31.0,"[0.4496287702713411, 0.45470916748967843, 0.45..."


In [10]:
ride_max = np.array([max(p) for p in df_res["mean_path"]]) # max running-mean per ride (NCM)
sober_mask = (df_res["run"] == "Sober").values # mask to select sober rides
pids = df_res["participant_id"].values # participant IDs
detection_step = [] # detection steps

# loop rides
for i, path in enumerate(df_res["mean_path"]):
    cal = ride_max[sober_mask & (pids != pids[i])] # drop participants rides (LOPO)

    # fold-level recalibrated threshold: m-th largest cal value (conformal correction)
    m = int(np.floor(alpha * (len(cal) + 1)))
    tau = np.sort(cal)[-m] if m else np.inf # cal too small: never alarm

    # detection steps: windows where running-mean > tau; first one (if any) is the alarm
    alarm_steps = np.flatnonzero(np.asarray(path) > tau)
    detection_step.append(int(alarm_steps[0]) if len(alarm_steps) else None)

# Group results in a DF
df_res["detection_step"] = detection_step
df_res["detected"] = [s is not None for s in detection_step]
df_res["detection_time_s"] = [
    (s * pe_s + pe_w) / sampling_rate if s is not None else None for s in detection_step
]
df_res

,run,participant_id,mean_path,detection_step,detected,detection_time_s
0,High,3.0,"[0.4782833819480143, 0.5176180617936335, 0.553...",13.0,True,23.0
1,Sober,3.0,"[0.46058195079816233, 0.48201164664443574, 0.4...",NaN,False,NaN
2,High,3.0,"[0.4804965395454593, 0.5221958112488587, 0.556...",18.0,True,28.0
3,Low,3.0,"[0.4538851868405219, 0.46681271101042715, 0.48...",NaN,False,NaN
4,Low,3.0,"[0.47409700322973053, 0.5120840505749872, 0.54...",24.0,True,34.0
...,...,...,...,...,...,...
136,Sober,31.0,"[0.4337333199146217, 0.42564298076245216, 0.42...",NaN,False,NaN
137,Low,31.0,"[0.4874349185638229, 0.521108561581999, 0.5465...",15.0,True,25.0
138,Low,31.0,"[0.44833040445940886, 0.45153349454108144, 0.4...",NaN,False,NaN
139,Sober,31.0,"[0.4496287702713411, 0.45470916748967843, 0.45...",NaN,False,NaN


In [11]:
_ = bootstrap_report(df_res, glue=glue, per_ride_path=per_ride_path)

       support     detection_rate det_time_min det_time_q25 det_time_med det_time_q75 det_time_max                FAR
run                                                                                                                  
High        46  0.91 [0.82, 0.98]  21 [21, 21]  22 [21, 23]  25 [23, 27]  29 [26, 32]  36 [33, 36]                NaN
Low         50  0.50 [0.36, 0.64]  20 [20, 21]  22 [21, 27]  27 [24, 34]  35 [27, 47]  62 [47, 62]                NaN
Sober       45                NaN          NaN          NaN          NaN          NaN          NaN  0.02 [0.00, 0.07]

accuracy sober vs non-sober: 0.79 [0.72, 0.85]
accuracy sober vs high:      0.95 [0.90, 0.99]


/home/marco/git/online-impairment-detection/.venv/lib/python3.13/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)
